# Game Team Stats to Delta table

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType
from pyspark.sql.window import Window
from delta.tables import DeltaTable
import sys
sys.path.insert(0, "/Workspace/Repos")
from transforms import conform_game_team_stats

ACCOUNT    = "footballmanagerli"
SRC        = f"abfss://alsource@{ACCOUNT}.dfs.core.windows.net/game-team-stats/"
DEST       = f"abfss://aldestination@{ACCOUNT}.dfs.core.windows.net/game-team-stats/game_team_stats"
CHECKPOINT = f"abfss://aldestination@{ACCOUNT}.dfs.core.windows.net/_checkpoints/game_team_stats"
TABLE      = "nfl.game_team_stats"

In [ ]:
# Base schema for input - matches Azure's snake_case output
base_schema = StructType([
    StructField("game_id",      StringType()),
    StructField("team_id",      StringType()),
    StructField("team_name",    StringType()),
    StructField("team_abbr",    StringType()),
    StructField("home_away",    StringType()),
    # Stats are in the data but we let Spark infer additional fields
])

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS nfl")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABLE} (
    game_id                     STRING    COMMENT 'Game ID (part of composite key)',
    team_id                     STRING    COMMENT 'Team ID (part of composite key)',
    team_name                   STRING    COMMENT 'Team name',
    team_abbr                   STRING    COMMENT 'Team abbreviation',
    home_away                   STRING    COMMENT 'H=home, A=away',
    pass_completions            LONG,
    pass_attempts               LONG,
    pass_yards                  LONG,
    pass_touchdowns             LONG,
    interceptions               LONG,
    rush_attempts               LONG,
    rush_yards                  LONG,
    rush_touchdowns             LONG,
    receiving_yards             LONG,
    receiving_touchdowns        LONG,
    third_down_conversions      LONG,
    third_down_attempts         LONG,
    red_zone_scores             LONG,
    red_zone_attempts           LONG,
    first_downs                 LONG,
    possession_seconds          LONG    COMMENT 'Time of possession in seconds',
    total_yards                 LONG,
    fumbles                     LONG,
    fumbles_lost                LONG,
    sacks                       LONG,
    sack_yards                  LONG,
    interceptions_defense       LONG    COMMENT 'Defensive interceptions',
    _source_file                STRING    COMMENT 'Blob file path',
    _ingested_at                TIMESTAMP COMMENT 'When ingested into Delta'
)
USING DELTA
LOCATION '{DEST}'
""")

In [ ]:
# Read JSON and apply schema
raw = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("multiLine", "true")
    .schema(base_schema)
    .option("inferSchema", "true")  # Allow additional fields
    .load(SRC)
    .select(
        "*",
        F.col("_metadata.file_path").alias("_source_file"),
        F.col("_metadata.file_modification_time").alias("_snapshot_at"),
    ))

# Apply conform function, then add back metadata columns before writing stream
conformed = conform_game_team_stats(raw)
game_stats = (raw
    .select(
        *[F.col(c) for c in conformed.columns],  # All data columns from conform
        "_source_file",
        "_snapshot_at"
    )
    .withColumn("_ingested_at", F.current_timestamp())
)

In [ ]:
def upsert_batch(batch_df, batch_id):
    """Upsert with automatic schema evolution: add missing columns if source has new fields."""
    
    # Step 1: Dedupe within batch (keep newest snapshot per game/team combo)
    # Note: conform_game_team_stats() strips schema to target columns, so use _ingested_at, not _snapshot_at
    w = Window.partitionBy("game_id", "team_id").orderBy(F.col("_ingested_at").desc())
    latest = (batch_df
        .withColumn("_rn", F.row_number().over(w))
        .filter("_rn = 1")
        .drop("_rn"))
    
    # Step 2: Auto-add missing columns (schema evolution)
    target_table = DeltaTable.forName(spark, TABLE)
    target_schema = {field.name: field.dataType for field in target_table.toDF().schema}
    batch_columns = {field.name for field in latest.schema}
    
    for col_name, col_type in target_schema.items():
        if col_name not in batch_columns:
            latest = latest.withColumn(col_name, F.lit(None).cast(col_type))
    
    # Step 3: Merge into table
    (target_table.alias("t")
        .merge(latest.alias("s"), 
               "t.game_id = s.game_id AND t.team_id = s.team_id")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())

(game_stats.writeStream
    .foreachBatch(upsert_batch)
    .option("checkpointLocation", CHECKPOINT)
    .trigger(availableNow=True)
    .start()
    .awaitTermination())

In [ ]:
%sql
SELECT COUNT(*) AS rows FROM nfl.game_team_stats;

SELECT COUNT(DISTINCT game_id) AS unique_games FROM nfl.game_team_stats;

SELECT game_id, team_id, team_name, home_away, pass_attempts, pass_completions, possession_seconds 
FROM nfl.game_team_stats ORDER BY game_id DESC LIMIT 10;